# Agent Platform Training でのカスタムコンテナの使用

**学習目標:**
1. BigQuery を使用してトレーニング用と検証用のデータ分割を作成する方法を学ぶ
1. 機械学習モデルを Docker コンテナにパッケージ化し、Agent Platform 上でトレーニングする方法を学ぶ
1. Agent Platform 上のハイパーパラメータ調整エンジンを使用して最適なハイパーパラメータを見つける方法を学ぶ
1. トレーニング済みの機械学習モデルを REST API として Agent Platform にデプロイし、予測リクエストを送信する方法を学ぶ

このラボでは、地図測量データから森林被覆タイプ（Cover Type）を予測するマルチクラス分類モデルをトレーニングするアプリケーションを開発し、Docker イメージとしてパッケージ化して、**Agent Platform Training** 上で実行します。このラボで使用する[データセット](../../../datasets/covertype/README.md)は、UCI Machine Learning Repository の **Covertype Data Set** に基づいています。

トレーニングコードでは、データの前処理とモデリングに `scikit-learn` を使用します。このコードは **Agent Platform** のハイパーパラメータ調整で使用できるように、`hypertune` パッケージを使用して実装されています。

In [ ]:
import os
import time

import pandas as pd
from google.cloud import aiplatform, bigquery
from google.cloud.aiplatform import hyperparameter_tuning as hpt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

%load_ext google.cloud.bigquery

## 環境設定の構成

ロケーションパス、接続文字列、およびその他の環境設定を指定します。ラボ環境を反映した設定で `REGION` および `ARTIFACT_STORE` を必ず更新してください。

- `REGION` - Agent Platform Training および Prediction のコンピューティングリージョン。
- `ARTIFACT_STORE` - 同じリージョンに作成された GCS バケット。

In [ ]:
REGION = "us-central1"

PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0]

ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"

DATA_ROOT = f"{ARTIFACT_STORE}/data"
JOB_DIR_ROOT = f"{ARTIFACT_STORE}/jobs"
TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"
API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"

In [ ]:
os.environ["JOB_DIR_ROOT"] = JOB_DIR_ROOT
os.environ["TRAINING_FILE_PATH"] = TRAINING_FILE_PATH
os.environ["VALIDATION_FILE_PATH"] = VALIDATION_FILE_PATH
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

`ARTIFACT_STORE` バケットが存在しない場合は作成します。このバケットは変数 `REGION` で指定されたリージョンに作成する必要があることに注意してください（すでに `REGION` とは異なるリージョンにこの名前のバケットが存在する場合は、下のセルのコマンドで `REGION` にバケットを再作成できるように `ARTIFACT_STORE` の名前を変更してください）。

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

## BigQuery へのデータセットのインポート

In [ ]:
%%bash

DATASET_LOCATION=US
DATASET_ID=covertype_dataset
TABLE_ID=covertype
DATA_SOURCE=gs://asl-public-data/data/covertype/dataset.csv
SCHEMA=Elevation:INTEGER,\
Aspect:INTEGER,\
Slope:INTEGER,\
Horizontal_Distance_To_Hydrology:INTEGER,\
Vertical_Distance_To_Hydrology:INTEGER,\
Horizontal_Distance_To_Roadways:INTEGER,\
Hillshade_9am:INTEGER,\
Hillshade_Noon:INTEGER,\
Hillshade_3pm:INTEGER,\
Horizontal_Distance_To_Fire_Points:INTEGER,\
Wilderness_Area:STRING,\
Soil_Type:STRING,\
Cover_Type:INTEGER

bq --location=$DATASET_LOCATION --project_id=$PROJECT_ID mk --dataset $DATASET_ID

bq --project_id=$PROJECT_ID --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

## Covertype データセットの探索

In [ ]:
%%bigquery
SELECT *
FROM `covertype_dataset.covertype`

## トレーニング用と検証用の分割の作成

BigQuery を使用してトレーニング用と検証用の分割をサンプリングし、GCS ストレージに保存します。
### トレーニング用の分割を作成

In [ ]:
!bq query \
-n 0 \
--destination_table covertype_dataset.training \
--replace \
--use_legacy_sql=false \
'SELECT * \
FROM `covertype_dataset.covertype` AS cover \
WHERE \
MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(cover))), 10) IN (1, 2, 3, 4)' 

In [ ]:
!bq extract \
--destination_format CSV \
covertype_dataset.training \
$TRAINING_FILE_PATH

### 検証用の分割を作成

## 演習

In [ ]:
# TODO: BQ テーブルの検証分割を作成するコード

In [ ]:
# TODO: 検証テーブルを GCS にエクスポートするコード

In [ ]:
df_train = pd.read_csv(TRAINING_FILE_PATH)
df_validation = pd.read_csv(VALIDATION_FILE_PATH)
print(df_train.shape)
print(df_validation.shape)

## トレーニングアプリケーションの開発

### `sklearn` トレーニングパイプラインの構成

トレーニングパイプラインは、`sklearn.preprocessing.StandardScaler` を使用してすべての数値特徴量を標準化し、`sklearn.preprocessing.OneHotEncoder` を使用してすべてのカテゴリ特徴量をエンコードすることによってデータを前処理します。モデリングには確率的勾配降下法（SGD）による線形分類器（`SGDClassifier`）を使用します。

In [ ]:
numeric_feature_indexes = slice(0, 10)
categorical_feature_indexes = slice(10, 12)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_feature_indexes),
        ("cat", OneHotEncoder(), categorical_feature_indexes),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", SGDClassifier(loss="log_loss", tol=1e-3)),
    ]
)

### すべての数値特徴量を `float64` に変換

`StandardScaler` からの警告メッセージを回避するために、すべての数値特徴量を `float64` に変換します。

In [ ]:
num_features_type_map = {
    feature: "float64" for feature in df_train.columns[numeric_feature_indexes]
}

df_train = df_train.astype(num_features_type_map)
df_validation = df_validation.astype(num_features_type_map)

### パイプラインをローカルで実行

In [ ]:
X_train = df_train.drop("Cover_Type", axis=1)
y_train = df_train["Cover_Type"]
X_validation = df_validation.drop("Cover_Type", axis=1)
y_validation = df_validation["Cover_Type"]

pipeline.set_params(classifier__alpha=0.001, classifier__max_iter=200)
pipeline.fit(X_train, y_train)

### トレーニング済みモデルの正解率（Accuracy）を計算

In [ ]:
accuracy = pipeline.score(X_validation, y_validation)
print(accuracy)

### ハイパーパラメータ調整アプリケーションの準備
このデータセットでのトレーニング実行は計算コストが高いため、Agent Platform Training 上で分散ハイパーパラメータ調整ジョブを実行することが有益です。

In [ ]:
TRAINING_APP_FOLDER = "training_app"
os.makedirs(TRAINING_APP_FOLDER, exist_ok=True)

### チューニングスクリプトの作成

`accuracy` 最適化指標を Agent Platform ハイパーパラメータ調整サービスに報告するために `hypertune` パッケージを使用している点に注目してください。

In [ ]:
%%writefile {TRAINING_APP_FOLDER}/train.py
import os
import subprocess
import sys

import fire
import hypertune
import numpy as np
import pandas as pd
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def train_evaluate(job_dir, training_dataset_path, validation_dataset_path, alpha, max_iter, hptune):
    
    df_train = pd.read_csv(training_dataset_path)
    df_validation = pd.read_csv(validation_dataset_path)

    if not hptune:
        df_train = pd.concat([df_train, df_validation])

    numeric_feature_indexes = slice(0, 10)
    categorical_feature_indexes = slice(10, 12)

    preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_feature_indexes),
        ('cat', OneHotEncoder(), categorical_feature_indexes) 
    ])

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SGDClassifier(loss='log_loss',tol=1e-3))
    ])

    num_features_type_map = {feature: 'float64' for feature in df_train.columns[numeric_feature_indexes]}
    df_train = df_train.astype(num_features_type_map)
    df_validation = df_validation.astype(num_features_type_map) 

    print('Starting training: alpha={}, max_iter={}'.format(alpha, max_iter))
    X_train = df_train.drop('Cover_Type', axis=1)
    y_train = df_train['Cover_Type']

    pipeline.set_params(classifier__alpha=alpha, classifier__max_iter=int(max_iter))
    pipeline.fit(X_train, y_train)

    if hptune:
        X_validation = df_validation.drop('Cover_Type', axis=1)
        y_validation = df_validation['Cover_Type']
        accuracy = pipeline.score(X_validation, y_validation)
        print('Model accuracy: {}'.format(accuracy))
        # hypertune で記録
        hpt = hypertune.HyperTune()
        hpt.report_hyperparameter_tuning_metric(
          hyperparameter_metric_tag='accuracy',
          metric_value=accuracy
        )

    # モデルを保存
    if not hptune:
        model_filename = 'model.pkl'
        with open(model_filename, 'wb') as model_file:
            pickle.dump(pipeline, model_file)
        gcs_model_path = "{}/{}".format(job_dir, model_filename)
        subprocess.check_call(['gcloud', 'storage', 'cp', model_filename, gcs_model_path], stderr=sys.stdout)
        print("Saved model in: {}".format(gcs_model_path)) 
    
if __name__ == "__main__":
    fire.Fire(train_evaluate)

### スクリプトを Docker イメージにパッケージ化

トレーニングイメージには、特定のバージョンの `scikit-learn` および `pandas` をインストールしている点に注目してください。これは、トレーニングコンテナ内のトレーニングランタイムが、サービングコンテナ内のサービングランタイムと確実に一致するようにするために行われます。

ベースイメージの URI がプロジェクトの **Artifact Registry** を指すように更新されていることを確認してください。

### 演習

コンテナが起動したときに 'train.py' ファイルが `/app` にコピーされ、実行されるように以下の Dockerfile を完成させてください。

In [ ]:
%%writefile {TRAINING_APP_FOLDER}/Dockerfile

FROM us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0
RUN pip install -U fire cloudml-hypertune scikit-learn==1.2.2

# TODO

### Docker イメージのビルド

**Cloud Build** を使用してイメージをビルドし、プロジェクトの **Artifact Registry** に push します。リモートのクラウドサービスを使用してイメージをビルドするため、ローカルに Docker をインストールする必要はありません。

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image"
IMAGE_TAG = "latest"
IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"

os.environ["IMAGE_URI"] = IMAGE_URI

In [ ]:
!gcloud builds submit --tag $IMAGE_URI $TRAINING_APP_FOLDER

## Agent Platform ハイパーパラメータ調整ジョブの送信

### ハイパーパラメータ構成ファイルの作成
トレーニングコードでは `SGDClassifier` を使用していることを思い出してください。トレーニングアプリケーションは、`SGDClassifier` を制御する次の 2 つのハイパーパラメータを受け取るように設計されています:
- Max iterations（最大イテレーション数）
- Alpha

以下のファイルは、最大 5 つの試行（トライアル）を並列実行し、`max_iter` については 2 つの離散値から、`alpha` については `1.0e-4` と `1.0e-1` の間の範囲から選択するように Agent Platform ハイパーパラメータ調整を構成します。

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_tuning_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

### 演習

ハイパーパラメータ調整エンジンを以下のパラメータ値で構成するように、以下の `HyperparameterTuningJob` コマンドを完成させてください:

* `max_iter`: 2 つの値 10 および 20
* `alpha`: 1.0e-4 から 1.0e-1 までの線形範囲の値

また、最大試行数（max trial count）と最大並列試行数（max parallel trials）をそれぞれ 5 に設定してハイパーパラメータ調整ジョブを開始するように `gcloud` コマンドを完成させてください。

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_tuning_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--hptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "e2-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

In [ ]:
hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=JOB_NAME,
    custom_job=custom_job,
    metric_spec={
        "accuracy": "maximize",
    },
    parameter_spec={...},  # TODO
    max_trial_count=...,  # TODO
    parallel_trial_count=...,  # TODO
)

hpt_job.run()

**注**: 参考として、YAML ファイルで構成を提供できる同等の gcloud ai コマンドを以下に示します。


```bash
MACHINE_TYPE="e2-standard-4"
REPLICA_COUNT=1
CONFIG_YAML=config.yaml

cat <<EOF > $CONFIG_YAML
studySpec:
  metrics:
  - metricId: accuracy
    goal: MAXIMIZE
  parameters:
  - parameterId: max_iter
    discreteValueSpec:
      values:
      - 10
      - 20
  - parameterId: alpha
    doubleValueSpec:
      minValue: 1.0e-4
      maxValue: 1.0e-1
    scaleType: UNIT_LINEAR_SCALE
  algorithm: ALGORITHM_UNSPECIFIED # ベイズ最適化を行う
trialJobSpec:
  workerPoolSpecs:  
  - machineSpec:
      machineType: $MACHINE_TYPE
    replicaCount: $REPLICA_COUNT
    containerSpec:
      imageUri: $IMAGE_URI
      args:
      - --job_dir=$JOB_DIR
      - --training_dataset_path=$TRAINING_FILE_PATH
      - --validation_dataset_path=$VALIDATION_FILE_PATH
      - --hptune
EOF

gcloud ai hp-tuning-jobs create     --region=$REGION     --display-name=$JOB_NAME     --config=$CONFIG_YAML     --max-trial-count=5     --parallel-trial-count=5

echo "JOB_NAME: $JOB_NAME"
```

Agent Platform Training ダッシュボードに移動し、「Hyperparameter Tuning Jobs（ハイパーパラメータ調整ジョブ）」で HP 調整ジョブの進行状況を確認します。

### HP 調整結果の取得

ジョブが完了したら、GCP コンソールを使用するか、以下の関数を使用してプログラムで結果を確認できます（このコードは、ハイパーパラメータ調整エンジンが最適化する指標が最大化されることを想定している点に注意してください）:

## 演習

`hpt_job` から最適なハイパーパラメータを取得するように、以下の関数の本文を完成させてください:

In [ ]:
def get_best_hyperparameters(hpt_job):
    metrics = [
        trial.final_measurement.metrics[0].value for trial in hpt_job.trials
    ]
    best_trial = hpt_job.trials[metrics.index(max(metrics))]
    best_parameters = ...  # TODO
    return best_parameters["alpha"], best_parameters["max_iter"]


alpha, max_iter = get_best_hyperparameters(hpt_job)
print(f"best alpha: {alpha}, best max_iter: {max_iter}")

## 最適なハイパーパラメータでモデルを再トレーニング

最適なハイパーパラメータを使用し、トレーニングと検証の分割を結合したものをトレーニングデータセットとして使用して、モデルを再トレーニングできるようになりました。

### トレーニングジョブの構成と実行

In [ ]:
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"forestcover_training_{TIMESTAMP}"
JOB_DIR = f"{JOB_DIR_ROOT}/{JOB_NAME}"

args = [
    f"--job_dir={JOB_DIR}",
    f"--training_dataset_path={TRAINING_FILE_PATH}",
    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
    f"--alpha={alpha}",
    f"--max_iter={max_iter}",
    f"--nohptune",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "e2-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{JOB_DIR}/staging",
)

custom_job.run()

**注**: 参考として、コンテナトレーニングを実行するための同等の bash コマンドを以下に示します。

Python セルで環境変数を設定します。
```python
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
os.environ["alpha"] = alpha
os.environ["max_iter"] = max_iter
os.environ["TIMESTAMP"] = TIMESTAMP
```

以下の bash コマンドを実行します

```bash
JOB_NAME=forestcover_training_$TIMESTAMP
JOB_DIR=$JOB_DIR_ROOT/$JOB_NAME

MACHINE_TYPE=e2-standard-4
REPLICA_COUNT=1

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,replica-count=$REPLICA_COUNT,container-image-uri=$IMAGE_URI"

ARGS="--job_dir=$JOB_DIR,--training_dataset_path=$TRAINING_FILE_PATH,--validation_dataset_path=$VALIDATION_FILE_PATH,--alpha=$alpha,--max_iter=$max_iter,--nohptune"

gcloud ai custom-jobs create   --region=$REGION   --display-name=$JOB_NAME   --worker-pool-spec=$WORKER_POOL_SPEC   --args=$ARGS
```

### トレーニング出力の確認

トレーニングスクリプトは、トレーニング済みモデルを GCS 上の `JOB_DIR` フォルダに 'model.pkl' として保存しました。

In [ ]:
!gcloud storage ls $JOB_DIR

## Agent Platform Prediction へのモデルのデプロイ

In [ ]:
MODEL_NAME = "forest_cover_classifier"
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)
SERVING_MACHINE_TYPE = "e2-standard-2"

### トレーニング済みモデルのアップロード

## 演習

`aiplatform.Model.upload` を使用してトレーニング済みモデルをアップロードしてください:

In [ ]:
uploaded_model = # TODO

### アップロードしたモデルのデプロイ

## 演習

`uploaded_model` を使用してモデルをデプロイしてください:

In [ ]:
endpoint = # TODO

### 予測の提供
#### JSON 形式のインスタンスを含む入力ファイルを準備します。

## 演習

`endpoint` を使用してデプロイされたモデルにクエリを実行してください:

In [ ]:
instance = [
    2841.0,
    45.0,
    0.0,
    644.0,
    282.0,
    1376.0,
    218.0,
    237.0,
    156.0,
    1003.0,
    "Commanche",
    "C4758",
]

# TODO

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.